# CEG-WM Stage-A HF-v2 rank-gate confirmation

Run all cells after setting Colab Secrets `CEG_WM_ROOT_KEY` and `HF_TOKEN`. The notebook resolves the HF-v2 rank-gate branch head once, checks it out detached, and automatically resumes only a validated compatible checkpoint. It runs the frozen untouched identity confirmation roster and hands off the completed Drive ZIP/checksum pair without interpreting it. LPIPS and attacks remain unmeasured, and the returned package awaits independent validation and Agent5 adjudication.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
from pathlib import Path
import os
def _required_secret(name):
    value = os.environ.pop(name, None)
    if value is None:
        value = userdata.get(name)
    if not isinstance(value, str) or not value.strip():
        raise RuntimeError(f'missing required Colab Secret: {name}')
    return value
root_key = _required_secret('CEG_WM_ROOT_KEY')
hf_token = _required_secret('HF_TOKEN')
run_store_root = Path('/content/drive/MyDrive/CEG-WM/stage_a_hf_v2_rankgate')
run_store_root.mkdir(parents=True, exist_ok=True)


In [ ]:
import json, re, subprocess, sys
repo = Path('/content/CEG-WM-stage-a-hf-v2-rankgate-exact')
if repo.exists():
    raise RuntimeError('detached checkout path already exists')
subprocess.run(['git', 'init', str(repo)], check=True)
subprocess.run(['git', '-C', str(repo), 'remote', 'add', 'origin', 'https://github.com/RICHAAARC/CEG-WM.git'], check=True)
subprocess.run(['git', '-C', str(repo), 'fetch', '--depth', '1', 'origin', 'refs/heads/stage-a-hf-v2-rankgate'], check=True)
subprocess.run(['git', '-C', str(repo), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
resolved_exact = subprocess.run(['git', '-C', str(repo), 'rev-parse', 'HEAD'], check=True, capture_output=True, text=True).stdout.strip()
if re.fullmatch(r'[0-9a-f]{40}', resolved_exact) is None:
    raise RuntimeError('resolved Stage-A branch head is not an exact revision')
if subprocess.run(['git', '-C', str(repo), 'status', '--porcelain'], check=True, capture_output=True, text=True).stdout:
    raise RuntimeError('execution checkout is not clean')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(repo)], check=True)


In [ ]:
local_output_root = Path('/content/cegwm-stage-a-hf-v2-rankgate-local')
runner_env = dict(os.environ)
runner_env['CEG_WM_ROOT_KEY'] = root_key
runner_env['HF_TOKEN'] = hf_token
command = [sys.executable, '-m', 'experiments.stage_a.run_hf_a2_colab', '--repo-root', str(repo), '--output-root', str(local_output_root), '--expected-exact', resolved_exact, '--run-store-root', str(run_store_root)]
run_id = None
fatal_event = None
try:
    process = subprocess.Popen(command, cwd=str(repo), env=runner_env, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True)
    for line in process.stdout:
        if line.startswith('CEGWM_PROGRESS '):
            progress = json.loads(line.removeprefix('CEGWM_PROGRESS '))
            if not set(progress).issubset({'run_id', 'committed', 'fixed_total', 'phase'}):
                raise RuntimeError('runner progress exposed unexpected fields')
            candidate_run_id = progress.get('run_id')
            if re.fullmatch(r'a2hfv2-[0-9a-f]{24}', candidate_run_id or '') is None:
                raise RuntimeError('runner progress has invalid deterministic run identity')
            if run_id is not None and run_id != candidate_run_id:
                raise RuntimeError('runner changed deterministic run identity')
            run_id = candidate_run_id
            print({'run_id': run_id, 'committed': progress['committed'], 'fixed_total': progress['fixed_total']})
        elif line.startswith('CEGWM_FATAL '):
            fatal_event = json.loads(line.removeprefix('CEGWM_FATAL '))
            if set(fatal_event) != {'run_id', 'error_class', 'export_status'}:
                raise RuntimeError('runner fatal event exposed unexpected fields')
            candidate_run_id = fatal_event.get('run_id')
            if re.fullmatch(r'a2hfv2-[0-9a-f]{24}', candidate_run_id or '') is None:
                raise RuntimeError('runner fatal event has invalid deterministic run identity')
            if run_id is not None and run_id != candidate_run_id:
                raise RuntimeError('runner changed deterministic run identity')
            run_id = candidate_run_id
    runner_rc = process.wait()
finally:
    runner_env.pop('CEG_WM_ROOT_KEY', None)
    runner_env.pop('HF_TOKEN', None)
    root_key = hf_token = ''
    del root_key, hf_token, runner_env
if run_id is None:
    raise RuntimeError('runner produced no deterministic run identity')


In [ ]:
drive_run_dir = run_store_root / run_id
if runner_rc == 2:
    if fatal_event is None:
        raise RuntimeError('runner RC2 has no bounded failure event')
    error_class = fatal_event['error_class']
    if error_class not in {'initialization_failure', 'resume_validation_failure', 'runtime_execution_failure', 'checkpoint_failure', 'final_export_failure'}:
        raise RuntimeError('runner RC2 error class is not predeclared')
    zip_path = drive_run_dir / (f'{run_id}.zip' if error_class == 'final_export_failure' else f'failure-{error_class}.zip')
else:
    if fatal_event is not None or runner_rc not in {0, 1}:
        raise RuntimeError('runner terminal event/RC mismatch')
    zip_path = drive_run_dir / f'{run_id}.zip'
checksum_path = drive_run_dir / f'{zip_path.name}.sha256'
pair_present = zip_path.is_file() and checksum_path.is_file()
summary = {'run_id': run_id, 'resolved_exact': resolved_exact, 'runner_rc': runner_rc, 'zip_path': str(zip_path), 'checksum_path': str(checksum_path), 'pair_present': pair_present}
print(summary)
if not pair_present:
    raise RuntimeError('runner did not leave a complete terminal package pair for external validation')
if runner_rc != 0:
    raise RuntimeError('runner completed with retained operational failures')
